In [1]:
import os
import json
from ast import literal_eval
from datetime import datetime, timedelta

import pandas as pd
import numpy as np

from python_utilities.db_connection import DbConnection

In [2]:
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')

INFO [2026-01-15 11:43:11] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


In [3]:
text = 'diese kostenrechnung kann schriftlich oder zu protokoll der geschäftsstelle erinnerung bei'

In [4]:
data = analytics_db.sql_to_df(f"""
    SELECT *
    FROM llm_attachments_predictions
    WHERE subtype = 'debtor_name' AND LOWER(value)="'{text.lower()}'"
""")


In [5]:
data

,id,created_at,model_name,type,subtype,value,attachment_id
0,1824748,2025-11-22 09:14:36,aftercourt_classification,aftercourt_classification,debtor_name,'Diese Kostenrechnung Kann Schriftlich Oder Zu...,44529743
1,1827521,2025-11-22 09:16:38,aftercourt_classification,aftercourt_classification,debtor_name,'Diese Kostenrechnung Kann Schriftlich Oder Zu...,44531561
2,1834760,2025-11-23 10:27:44,aftercourt_classification,aftercourt_classification,debtor_name,'Diese Kostenrechnung Kann Schriftlich Oder Zu...,44689202
3,1839169,2025-11-23 17:40:39,aftercourt_classification,aftercourt_classification,debtor_name,'Diese Kostenrechnung Kann Schriftlich Oder Zu...,44707482
4,1841227,2025-11-24 07:10:44,aftercourt_classification,aftercourt_classification,debtor_name,'Diese Kostenrechnung Kann Schriftlich Oder Zu...,44708172
...,...,...,...,...,...,...,...
215,2950249,2026-01-14 15:29:42,aftercourt_classification,aftercourt_classification,debtor_name,'Diese Kostenrechnung Kann Schriftlich Oder Zu...,24825525155740-3
216,2959663,2026-01-14 18:15:56,aftercourt_classification,aftercourt_classification,debtor_name,'Diese Kostenrechnung Kann Schriftlich Oder Zu...,49747921
217,2959952,2026-01-14 18:16:08,aftercourt_classification,aftercourt_classification,debtor_name,'Diese Kostenrechnung Kann Schriftlich Oder Zu...,49775489
218,2959974,2026-01-14 22:53:38,aftercourt_classification,aftercourt_classification,debtor_name,'Diese Kostenrechnung Kann Schriftlich Oder Zu...,49778889


In [6]:
attachment_ids = data.attachment_id.unique()

In [7]:
len(attachment_ids)

218

Get textraxt texts for them

In [8]:
attachment_ids = list(attachment_ids)

In [9]:
textract = analytics_db.sql_to_df(f"""
    SELECT *
    FROM textract_jobs
    WHERE attachment_id IN {tuple(attachment_ids)}
    AND status = 'SUCCEEDED'
""")

# drop null values
textract = textract.dropna(subset=['s3_link'])
# drop dublicates
textract = textract.drop_duplicates(subset=['s3_link'])
s3_links = textract['s3_link'].tolist()

In [10]:
textract

,id,job_id,status,s3_link,created_at,updated_at,attachment_id
0,1229165,a134bfdb544b262b076969148ba98cf91027b26461a03b...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-12-02 11:25:07,2025-12-02 11:50:38,23973761582236-1
1,1349576,7c41fc7ddef913328e45c541aec3ac1226fb5a8e293b9c...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-08 08:53:20,2026-01-08 09:06:24,24693116189212-1
2,1376715,b5abb5c4f3b7abc8f511c5fc46390ed0ae5095dec66dad...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 11:13:09,2026-01-14 11:36:06,24823927696284-3
3,1376977,060dc51686ebf6a9efbb0c6338c2d08a3fd5f85a1a7520...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 11:53:35,2026-01-14 12:07:02,24824722387996-2
4,1377138,e6256f174223151446b1951bcf577064f56fd33a4c6668...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 12:22:58,2026-01-14 12:35:15,24825525155740-3
...,...,...,...,...,...,...,...
214,1376767,9e5256ad6a100c35d5b8b34be4302c0f7c23552531324e...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 11:13:35,2026-01-14 11:38:36,49736954
215,1378119,6b2f1f0549676fb533d969eed0852c32454200179703c4...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 14:32:46,2026-01-14 14:44:39,49747921
216,1378656,cd75c99304472ade02ed5eb096870c15617dffe4a6e7f2...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 16:13:17,2026-01-14 16:26:23,49775489
217,1378811,43a0f99f2602dca4009445f3aea954a3da55cc6411ae25...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 17:12:57,2026-01-14 17:24:20,49778889


In [11]:
import boto3
def get_texts_from_textract_outputs(
        textract_jsons):
        return [
            "\n".join([d["Text"] for d in doc if d["BlockType"] == "LINE"])
            for doc in textract_jsons
        ]

def get_jsons_from_s3(s3_links):
    """
    Retrieves JSON objects from given S3 addresses.
    """
    # create S3 client with specific user profile
    session = boto3.Session(profile_name='739275445236_DataScienceUser')
    s3_client = session.client('s3')
    json_list = []

    for link in s3_links:
        json_object = [{"BlockType": "LINE", "Text": ""}]
        try:
            # Parse the S3 URI
            parts = link.replace("s3://", "").split("/")
            bucket_name = parts[0]
            key = "/".join(parts[1:])
            # Get the object from S3
            response = s3_client.get_object(Bucket=bucket_name, Key=key)

            # Read the content and parse JSON
            content = response['Body'].read().decode('utf-8')
            json_object = json.loads(content)
        except Exception as e:
            print(f"Error from {link} {e}")
            
        json_list.append(json_object)
    return json_list

returned_list=get_jsons_from_s3(s3_links)
texts = get_texts_from_textract_outputs(returned_list)

INFO [2026-01-15 11:43:13] - Found credentials in shared credentials file: ~/.aws/credentials


In [12]:
textract['texts'] = texts

In [13]:
textract

,id,job_id,status,s3_link,created_at,updated_at,attachment_id,texts
0,1229165,a134bfdb544b262b076969148ba98cf91027b26461a03b...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2025-12-02 11:25:07,2025-12-02 11:50:38,23973761582236-1,ANNETTE ZIMMER\nDie nachstehend gewählten Form...
1,1349576,7c41fc7ddef913328e45c541aec3ac1226fb5a8e293b9c...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-08 08:53:20,2026-01-08 09:06:24,24693116189212-1,Sven Wiegand\nDie nachstehend gewählten Formul...
2,1376715,b5abb5c4f3b7abc8f511c5fc46390ed0ae5095dec66dad...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 11:13:09,2026-01-14 11:36:06,24823927696284-3,JÖRG SCHMIDT\nDie nachstehend gewählten Formul...
3,1376977,060dc51686ebf6a9efbb0c6338c2d08a3fd5f85a1a7520...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 11:53:35,2026-01-14 12:07:02,24824722387996-2,Sven Wiegand\nDie nachstehend gewählten Formul...
4,1377138,e6256f174223151446b1951bcf577064f56fd33a4c6668...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 12:22:58,2026-01-14 12:35:15,24825525155740-3,Sven Wiegand\nDie nachstehend gewählten Formul...
...,...,...,...,...,...,...,...,...
214,1376767,9e5256ad6a100c35d5b8b34be4302c0f7c23552531324e...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 11:13:35,2026-01-14 11:38:36,49736954,STEFAN HOFFMANN\nDie nachstehend gewählten For...
215,1378119,6b2f1f0549676fb533d969eed0852c32454200179703c4...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 14:32:46,2026-01-14 14:44:39,49747921,Manuel Stember\nDie nachstehend gewählten Form...
216,1378656,cd75c99304472ade02ed5eb096870c15617dffe4a6e7f2...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 16:13:17,2026-01-14 16:26:23,49775489,S. Schmitz\nDie nachstehend gewählten Formulie...
217,1378811,43a0f99f2602dca4009445f3aea954a3da55cc6411ae25...,SUCCEEDED,s3://pair-data-engineering-new/ocr_prepared_ou...,2026-01-14 17:12:57,2026-01-14 17:24:20,49778889,Frank Lüers\nDie nachstehend gewählten Formuli...


In [15]:
data_save = textract[['attachment_id', 's3_link', 'texts']]
data_save.to_csv('/Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court/data/improvments_data/ladung_fp_protokoll_docs.csv', index=False)